## Repository path configuration
All repository files are accessed through relative paths. Run the notebook from the repository root or from the `notebooks/` directory.


In [ ]:
from pathlib import Path
import os

# Resolve the repository root whether Jupyter starts in the repository root
# or directly inside the notebooks directory.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

DATA_DIR = ROOT / "data"
EXTERNAL_DATA_DIR = DATA_DIR / "external"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
RESULTS_DIR = OUTPUTS_DIR / "results"
FIGURES_DIR = OUTPUTS_DIR / "figures"
EXTERNAL_MATERIALS_DIR = ROOT / "external_materials"
MODEL_WEIGHTS_DIR = EXTERNAL_MATERIALS_DIR / "model_weights"

for folder in [
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
    PROCESSED_DATA_DIR / "logs",
    RESULTS_DIR,
    FIGURES_DIR,
    MODEL_WEIGHTS_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)


In [ ]:
# ΟΠΤΙΚΟΠΟΙΗΣΗ LOSS ΓΙΑ BERT FAMILY MODELS και τα υπόλοιπα.

In [ ]:
import os
os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY", "")

In [ ]:
!pip install wandb -qU  # Εγκατάσταση της πιο πρόσφατης έκδοσης της βιβλιοθήκης WandB.
import wandb            # Εισαγωγή της WandB.

# Αυτή η εντολή ενεργοποιεί τον χρήστη να συνδεθεί στο λογαριασμό του WandB.
# Θα χρειαστεί ένα API Key.
wandb.login(key=os.environ["WANDB_API_KEY"]) if os.getenv("WANDB_API_KEY") else None

### Optional Google Colab use
Google Drive mounting is not required. In Colab, clone or upload the complete repository and run the notebook from the repository root. External model weights, when needed, must be placed under `external_materials/model_weights/`.


In [ ]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
!pip install transformers seaborn -q
!pip install umap-learn -q

import math
#import umap.umap_ as umap
import os
import random
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, MaxNLocator

In [ ]:
sns.set(style="whitegrid", font_scale=1.0)

print("Για ποιά μοντέλα θέλεις loss curves?")

print("Για Encoder models:")
print("Πληκτρολόγησε 1 για BERT, DeBERTa, XLNet, RoBERTa\n")
print("Πληκτρολόγησε 2 για ELECTRA, ALBERT, DistilBERT, SciBERT\n")

print("Για Decoder models:")
print("Πληκτρολόγησε 3 για DeepSeek, Falcon, GEMMA, LLaMA-3\n")
print("Πληκτρολόγησε 4 για GPT-2, GPT-Neo\n")

print("Πληκτρολόγησε 5 για CNNs\n")

option = int(input("Επιλογή: "))

if option == 1:
    loss_files = {
        "BERT":      "data/processed/logs/loss_BERT.csv",
        "DeBERTa":   "data/processed/logs/loss_DeBERTa.csv",
        "XLNet":     "data/processed/logs/loss_XLNet.csv",
        "RoBERTa":   "data/processed/logs/loss_RoBERTa.csv",
    }
elif option == 2:
    loss_files = {
        "ELECTRA":   "data/processed/logs/loss_ELECTRA.csv",
        "ALBERT":    "data/processed/logs/loss_ALBERT.csv",
        "DistilBERT":    "data/processed/logs/loss_DistilBERT.csv",
        "SciBERT":    "data/processed/logs/loss_SciBERT.csv",
    }
elif option == 3:
    loss_files = {
        "DeepSeek":      "data/processed/logs/loss_DeepSeek.csv",
        "Falcon":     "data/processed/logs/loss_Falcon.csv",
        "GEMMA":   "data/processed/logs/loss_GEMMA.csv",
        "LLaMA_3":    "data/processed/logs/loss_LLaMA_3.csv",
    }
elif option == 4:
    loss_files = {
        "GPT_2":   "data/processed/logs/loss_GPT_2.csv",
        "GPT_Neo":    "data/processed/logs/loss_GPT_Neo.csv",
    }
else:
    loss_files = {
        "CNN":   "data/processed/logs/loss_CNN.csv",
        "CNN_V2":   "data/processed/logs/loss_CNN_V2.csv",
    }

In [ ]:
train_color = "#1f77b4"   # Blue
val_color   = "#ff7f0e"   # Orange

model_names = list(loss_files.keys())
num_models = len(model_names)

cols = 2
rows = math.ceil(num_models / cols)

fig, axes = plt.subplots(rows, cols, figsize=(10, 4 * rows))
axes = axes.flatten()

for idx, model_name in enumerate(model_names):
    ax = axes[idx]
    df = pd.read_csv(loss_files[model_name])

    # Υποθέτουμε ότι το CSV έχει: epoch, train_loss, val_loss
    # Αν κάποια στήλη λείπει, μπορούμε να ελέγξουμε:
    has_val = "val_loss" in df.columns and df["val_loss"].notna().any()

    # TRAIN loss
    sns.lineplot(
        ax=ax,
        data=df,
        x="epoch",
        y="train_loss",
        marker="o",
        markersize=7,
        linewidth=2,
        color=train_color,
        label="Train"
    )

    # VALIDATION loss
    if has_val:
        sns.lineplot(
            ax=ax,
            data=df,
            x="epoch",
            y="val_loss",
            marker="X",
            markersize=7,
            linewidth=2,
            linestyle="--",
            color=val_color,
            label="Validation"
        )

    ax.set_title(model_name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")

    # Ακέραιοι στον x-άξονα
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    # Y-axis 0.0–1.0 με βήμα 0.2
    ax.set_ylim(0.0, 1.0)
    ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

    ax.grid(True)
    ax.legend(fontsize=11, loc="best")

# Αν περισσέψουν κενά subplots (π.χ. μονός αριθμός μοντέλων)
for j in range(num_models, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
if option == 1:
    fig.savefig(
        "outputs/figures/fig_comparable/BERT_family_loss_curves.pdf",
        format="pdf",
        bbox_inches="tight",
        dpi=300
    )
else:
    fig.savefig(
        f"outputs/figures/fig_comparable/Other_models_loss_curves.pdf",
        format="pdf",
        bbox_inches="tight",
        dpi=300
    )